# Definitive model — reading from Feast, and a real leakage fix that didn't go as planned

Task 6 of the plan: retrain the baseline from `notebooks/03_baseline_model.ipynb`, but reading features from Feast's offline store (task 5) instead of straight from `data/processed/features.parquet`, and — flagged as deferred back in task 3 — recomputing `district_avg_price_per_m2` leakage-free (fit on the train fold only, not the whole dataset).

That second part is most of this notebook. Fixing the leakage didn't just make the model "a bit more honest" — it broke it, in a way that took real investigation to understand. This notebook documents that investigation and the decision that came out of it, not just the final answer.

## Reading features from Feast's offline store

Reusing `ml/training/train.py`'s real functions (not reimplemented here) — this notebook's numbers must match what the script produces, same convention as `notebooks/03_baseline_model.ipynb`.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("..") / "ml" / "training"))
sys.path.insert(0, str(Path("..") / "ml" / "data_prep"))

from dotenv import load_dotenv
load_dotenv(Path("..") / ".env")

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import r2_score
import xgboost as xgb

from train import load_training_frame, to_categorical, CAT_COLS, TARGET_COL, TEST_SIZE, RANDOM_STATE
from build_features import smoothed_district_avg, SMOOTHING_K

features = load_training_frame()
print(f"Loaded {len(features)} rows from Feast's offline store")
print(features.columns.tolist())

Loaded 6811 rows from Feast's offline store
['id', 'price_usd', 'district', 'surface', 'property_type', 'operation_type']


In [2]:
# Sanity check against the parquet Feast's Postgres table was seeded from —
# full 6,811-row comparison already done once while building task 5 (see
# proyecto-mlops-plan.md); this is a quick re-confirmation, not the first time.
source = pd.read_parquet("../data/processed/features.parquet")
merged = source.merge(features, on="id", suffixes=("_source", "_feast"))
for col in ["district", "surface", "property_type", "operation_type"]:
    mismatches = (merged[f"{col}_source"] != merged[f"{col}_feast"]).sum()
    print(f"{col}: {mismatches} mismatches out of {len(merged)}")

district: 0 mismatches out of 6811
surface: 0 mismatches out of 6811
property_type: 0 mismatches out of 6811
operation_type: 0 mismatches out of 6811


## The leakage problem, revisited

`district_avg_price_per_m2` was flagged as a caveat since task 3: it's a leave-one-out mean, but fit on the **whole** dataset (train + test together, before any split). A test row's own price never leaks into its *own* feature value, but it can still reflect *other* test rows' prices through the shared district statistic — a real, if narrower, form of leakage. Task 3 explicitly deferred fixing this to "the definitive model" — this notebook.

The fix, in principle: fit the smoothed group average using **only the train fold**. Train rows keep their own leave-one-out value (nothing changes about the formula, just what data it's fit on). Test rows get the plain train-fold group average looked up by `(operation_type, district)` — there's nothing to leave out for them, since a test row's own price was never part of that statistic to begin with.

In [3]:
def fit_and_apply_district_avg(train_df, test_df, k=SMOOTHING_K):
    """Leakage-free district_avg_price_per_m2, fit on the train fold only.
    Train rows: their own LOO value (smoothed_district_avg, unchanged, fit
    on train_df alone). Test rows: plain train-fold group average, looked
    up by (operation_type, district), falling back to the train fold's
    global per-operation_type mean for unseen combinations.
    """
    train_df = train_df.copy()
    test_df = test_df.copy()

    train_df["district_avg_price_per_m2"] = smoothed_district_avg(train_df, k=k, district_col="district")

    price_per_m2 = train_df["price_usd"] / train_df["surface"]
    global_mean = price_per_m2.groupby(train_df["operation_type"]).mean()
    group_global_mean = train_df["operation_type"].map(global_mean)
    grp = price_per_m2.groupby([train_df["operation_type"], train_df["district"]])
    smoothed = (grp.transform("sum") + k * group_global_mean) / (grp.transform("count") + k)
    lookup_table = (
        train_df.assign(_smoothed=smoothed)
        .groupby(["operation_type", "district"])["_smoothed"]
        .first()
        .rename("district_avg_price_per_m2")
        .reset_index()
    )

    merged = test_df.merge(lookup_table, on=["operation_type", "district"], how="left")
    assert len(merged) == len(test_df), "merge changed row count"
    fallback = merged["operation_type"].map(global_mean)
    test_df["district_avg_price_per_m2"] = merged["district_avg_price_per_m2"].fillna(fallback).values
    return train_df, test_df


FEATURE_COLS_WITH_AVG = ["district", "surface", "property_type", "district_avg_price_per_m2"]
MODEL_PARAMS_TASK3 = dict(max_depth=5, min_child_weight=3)  # task 3's tuned config

sub = features[features["operation_type"] == "Venta"]
train_df, test_df = train_test_split(sub, test_size=TEST_SIZE, random_state=RANDOM_STATE)
train_df, test_df = fit_and_apply_district_avg(train_df, test_df)

X_train, X_test = train_df[FEATURE_COLS_WITH_AVG].copy(), test_df[FEATURE_COLS_WITH_AVG].copy()
y_train, y_test = train_df[TARGET_COL], test_df[TARGET_COL]
X_train, X_test = to_categorical(X_train, X_test, CAT_COLS)

model = xgb.XGBRegressor(enable_categorical=True, tree_method="hist", random_state=RANDOM_STATE, **MODEL_PARAMS_TASK3)
model.fit(X_train, np.log(y_train))
pred_test = np.exp(model.predict(X_test))
print("Venta test R2 with leakage-free feature, task 3's hyperparameters:", r2_score(y_test, pred_test))
print("(task 3 baseline, with the LEAKY version of this feature, same split: R2=0.7593)")

Venta test R2 with leakage-free feature, task 3's hyperparameters: -0.6068740804989738
(task 3 baseline, with the LEAKY version of this feature, same split: R2=0.7593)


A catastrophic drop, not a marginal one. Two questions before trusting this number: is the computation actually correct, and is this one split representative?

## Is this a bug?

Manually recomputing one train row's leave-one-out value and one test row's looked-up value by hand, independent of `fit_and_apply_district_avg`, and comparing.

In [4]:
# Test row: manual plain group average from the (unsplit-by-fit) train_df above
row = test_df.iloc[0]
d, op = row["district"], row["operation_type"]
train_grp = train_df[(train_df["operation_type"] == op) & (train_df["district"] == d)]
k = SMOOTHING_K
global_mean_op = (train_df[train_df["operation_type"] == op]["price_usd"] / train_df[train_df["operation_type"] == op]["surface"]).mean()
ppm = train_grp["price_usd"] / train_grp["surface"]
manual_test = (ppm.sum() + k * global_mean_op) / (len(train_grp) + k)
print("test row — function:", row["district_avg_price_per_m2"], " manual:", manual_test, " match:", abs(row["district_avg_price_per_m2"] - manual_test) < 1e-6)

# Train row: manual leave-one-out, excluding its own id from its own group
row = train_df.iloc[0]
d, op, own_id = row["district"], row["operation_type"], row["id"]
grp = train_df[(train_df["operation_type"] == op) & (train_df["district"] == d)]
others = grp[grp["id"] != own_id]
ppm = others["price_usd"] / others["surface"]
manual_train = (ppm.sum() + k * global_mean_op) / (len(others) + k)
print("train row — function:", row["district_avg_price_per_m2"], " manual:", manual_train, " match:", abs(row["district_avg_price_per_m2"] - manual_train) < 1e-3)

test row — function: 1139.9879440286682  manual: 1139.9879440286682  match: True
train row — function: 471.49767894876277  manual: 471.4976789487628  match: True


Both match by hand. The computation is correct — this isn't a bug in `fit_and_apply_district_avg`. So either the result is real, or it's an artifact of this one split. Task 3's own lesson ("Regularization" section) was exactly this: a single 80/20 split can look fine or catastrophic by chance — only multiple splits reveal the truth. Checking that here before drawing any conclusion.

In [5]:
FEATURE_COLS_NO_AVG = ["district", "surface", "property_type"]

for rs in [42, 1, 2, 3, 4]:
    train_df, test_df = train_test_split(sub, test_size=TEST_SIZE, random_state=rs)
    train_df_avg, test_df_avg = fit_and_apply_district_avg(train_df, test_df)

    X_train, X_test = train_df_avg[FEATURE_COLS_WITH_AVG].copy(), test_df_avg[FEATURE_COLS_WITH_AVG].copy()
    y_train, y_test = train_df_avg[TARGET_COL], test_df_avg[TARGET_COL]
    X_train, X_test = to_categorical(X_train, X_test, CAT_COLS)
    model = xgb.XGBRegressor(enable_categorical=True, tree_method="hist", random_state=42, **MODEL_PARAMS_TASK3)
    model.fit(X_train, np.log(y_train))
    r2_with = r2_score(y_test, np.exp(model.predict(X_test)))

    X_train2, X_test2 = train_df[FEATURE_COLS_NO_AVG].copy(), test_df[FEATURE_COLS_NO_AVG].copy()
    X_train2, X_test2 = to_categorical(X_train2, X_test2, CAT_COLS)
    model2 = xgb.XGBRegressor(enable_categorical=True, tree_method="hist", random_state=42, **MODEL_PARAMS_TASK3)
    model2.fit(X_train2, np.log(y_train))
    r2_without = r2_score(y_test, np.exp(model2.predict(X_test2)))

    print(f"random_state={rs}: R2 WITH leakage-free feature={r2_with:.4f}   WITHOUT the feature={r2_without:.4f}")

random_state=42: R2 WITH leakage-free feature=-0.6069   WITHOUT the feature=0.4925


random_state=1: R2 WITH leakage-free feature=0.1626   WITHOUT the feature=0.4345


random_state=2: R2 WITH leakage-free feature=0.3616   WITHOUT the feature=0.5909


random_state=3: R2 WITH leakage-free feature=-1.6511   WITHOUT the feature=0.4706


random_state=4: R2 WITH leakage-free feature=-0.1081   WITHOUT the feature=0.5542


Consistent, not a fluke: the honest feature is *worse than not having it at all* across every split tried, sometimes wildly so (down to R2=-1.65). Task 3's own precedent (`district_avg_price_per_m2` computed with full-dataset leave-one-out) was so informative it papered over what these hyperparameters (`max_depth=5, min_child_weight=3`) actually need: those were tuned via CV *against the leaky feature*. Once the feature gets honest — and therefore noisier, since it's now fit on a smaller train-only sample per district — the same hyperparameters let the model overfit to that noise instead of ignoring it.

Two ways forward: re-run the same CV process task 3 used, but against the leakage-free feature, to see if a stable hyperparameter config exists that still benefits from it — or drop the feature since `district` alone is already stable. Re-running CV first, since it doesn't foreclose on a feature that showed real signal before the leak was removed.

## Re-running CV against the leakage-free feature

Same method as task 3's "Regularization" section: 5-fold CV, `district_avg_price_per_m2` refit inside each fold (not just the final train/test split) so no fold ever sees its own validation rows' prices in the statistic.

In [6]:
def cv_scores(df_op, feature_cols, params, use_avg_feature, n_splits=5, seed=42):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    scores = []
    for train_idx, val_idx in kf.split(df_op):
        train_fold, val_fold = df_op.iloc[train_idx].copy(), df_op.iloc[val_idx].copy()
        if use_avg_feature:
            train_fold, val_fold = fit_and_apply_district_avg(train_fold, val_fold)
        X_train, X_val = train_fold[feature_cols].copy(), val_fold[feature_cols].copy()
        y_train, y_val = train_fold[TARGET_COL], val_fold[TARGET_COL]
        X_train, X_val = to_categorical(X_train, X_val, CAT_COLS)
        model = xgb.XGBRegressor(enable_categorical=True, tree_method="hist", random_state=42, **params)
        model.fit(X_train, np.log(y_train))
        scores.append(r2_score(y_val, np.exp(model.predict(X_val))))
    return np.array(scores)


candidates = [
    dict(max_depth=5, min_child_weight=3),
    dict(),
    dict(max_depth=3, min_child_weight=10),
    dict(max_depth=2, min_child_weight=10),
    dict(max_depth=2, min_child_weight=20),
    dict(max_depth=1, min_child_weight=10),
]

for operation_type in ["Venta", "Alquiler"]:
    df_op = features[features["operation_type"] == operation_type].reset_index(drop=True)
    print(f"=== {operation_type} (n={len(df_op)}), WITH leakage-free district_avg ===")
    for params in candidates:
        scores = cv_scores(df_op, FEATURE_COLS_WITH_AVG, params, use_avg_feature=True)
        print(f"  {params}: mean R2={scores.mean():.4f} std={scores.std():.4f}")

=== Venta (n=4777), WITH leakage-free district_avg ===


  {'max_depth': 5, 'min_child_weight': 3}: mean R2=-0.5989 std=1.4869


  {}: mean R2=-6.1402 std=12.3390


  {'max_depth': 3, 'min_child_weight': 10}: mean R2=0.2572 std=0.1468


  {'max_depth': 2, 'min_child_weight': 10}: mean R2=0.4236 std=0.0887


...

  {}: mean R2=0.0371 std=0.1361


  {'max_depth': 3, 'min_child_weight': 10}: mean R2=0.0071 std=0.5727


  {'max_depth': 2, 'min_child_weight': 10}: mean R2=0.2107 std=0.3360


  {'max_depth': 2, 'min_child_weight': 20}: mean R2=0.3290 std=0.1472


  {'max_depth': 1, 'min_child_weight': 10}: mean R2=0.3515 std=0.1171


Heavy regularization (`max_depth=1`, i.e. decision stumps) is the only region that looks stable with the feature included — but at that point the model is so constrained it barely uses any continuous feature's fine detail. Worth checking directly whether the feature is actually earning its keep there, or whether it's just neutral.

In [7]:
for operation_type in ["Venta", "Alquiler"]:
    df_op = features[features["operation_type"] == operation_type].reset_index(drop=True)
    print(f"=== {operation_type}, WITHOUT district_avg_price_per_m2 at all ===")
    for params in candidates:
        scores = cv_scores(df_op, FEATURE_COLS_NO_AVG, params, use_avg_feature=False)
        print(f"  {params}: mean R2={scores.mean():.4f} std={scores.std():.4f}")

=== Venta, WITHOUT district_avg_price_per_m2 at all ===


  {'max_depth': 5, 'min_child_weight': 3}: mean R2=0.5077 std=0.0486


  {}: mean R2=0.4915 std=0.0686


  {'max_depth': 3, 'min_child_weight': 10}: mean R2=0.5093 std=0.0520


  {'max_depth': 2, 'min_child_weight': 10}: mean R2=0.4999 std=0.0654


...

  {}: mean R2=0.0671 std=0.2841


  {'max_depth': 3, 'min_child_weight': 10}: mean R2=0.2678 std=0.1929


  {'max_depth': 2, 'min_child_weight': 10}: mean R2=0.2285 std=0.2629


  {'max_depth': 2, 'min_child_weight': 20}: mean R2=0.3350 std=0.1377


  {'max_depth': 1, 'min_child_weight': 10}: mean R2=0.3402 std=0.1176


Dropping the feature entirely is consistently as good or better than any hyperparameter config tried with it — at `max_depth=1` (where WITH the feature was stable) the WITHOUT numbers are effectively the same, and at less extreme regularization (`max_depth=2, min_child_weight=20`) WITHOUT clearly wins for both operation types (confirmed directly below, matched hyperparameters, same splits).

In [8]:
params = dict(max_depth=2, min_child_weight=20)
for operation_type in ["Venta", "Alquiler"]:
    df_op = features[features["operation_type"] == operation_type].reset_index(drop=True)
    with_scores = cv_scores(df_op, FEATURE_COLS_WITH_AVG, params, use_avg_feature=True)
    without_scores = cv_scores(df_op, FEATURE_COLS_NO_AVG, params, use_avg_feature=False)
    print(f"{operation_type}: WITH avg mean R2={with_scores.mean():.4f} std={with_scores.std():.4f}   "
          f"WITHOUT avg mean R2={without_scores.mean():.4f} std={without_scores.std():.4f}")

Venta: WITH avg mean R2=0.4225 std=0.1311   WITHOUT avg mean R2=0.5103 std=0.0581


Alquiler: WITH avg mean R2=0.3290 std=0.1472   WITHOUT avg mean R2=0.3350 std=0.1377


## Decision: drop `district_avg_price_per_m2` entirely

The feature's apparent value in task 3 came from leakage, not real signal beyond what `district` (categorical) already captures — once computed honestly, it never clearly earns its keep at any regularization level tested, and actively destabilizes the model at the regularization level that used to work. `max_depth=2, min_child_weight=20` is the new shared choice — best or near-best mean CV R2 for both operation types among everything tried, without the feature: Venta 0.51 (std 0.06), Alquiler 0.34 (std 0.14).

This is a real finding, not a compromise: the honest model is simpler (3 features instead of 4) *and* better-supported by the evidence than any version that keeps the leakage-prone feature.

`ml/feature_metadata.csv` and Feast's `arequipa_listings_features` feature view (task 5) still document/serve `district_avg_price_per_m2` — that stays useful at inference time (task 7): a genuinely new listing was never part of any training set, so there's no leakage concern serving the precomputed historical average there. It's specifically training/evaluation on the *original* dataset where reusing it is unsafe.

Encapsulated in `ml/training/train.py`: `FEATURE_COLS` reduced to `["district", "surface", "property_type"]`, `MODEL_PARAMS` updated to `max_depth=2, min_child_weight=20`.

In [9]:
from train import train_operation_model, save_model

for operation_type in ["Venta", "Alquiler"]:
    result = train_operation_model(features, operation_type)
    m = result["metrics"]
    print(
        f"\n{operation_type}: n_train={m['n_train']} n_test={m['n_test']}\n"
        f"  train: R2={m['train_r2']:.4f}  MAPE={m['train_mape_pct']:.1f}%\n"
        f"  test:  R2={m['r2']:.4f}  RMSE={m['rmse']:,.2f}  MAE={m['mae']:,.2f}  "
        f"MAPE={m['mape_pct']:.1f}%  (trivial baseline MAPE={m['naive_mean_mape_pct']:.1f}%)\n"
        f"  gap:   R2 diff={m['train_r2'] - m['r2']:.4f}   MAPE diff={m['mape_pct'] - m['train_mape_pct']:.1f}pp"
    )


Venta: n_train=3821 n_test=956
  train: R2=0.5759  MAPE=33.4%
  test:  R2=0.4596  RMSE=278,391.12  MAE=103,414.88  MAPE=41.3%  (trivial baseline MAPE=117.1%)
  gap:   R2 diff=0.1163   MAPE diff=7.8pp

Alquiler: n_train=1627 n_test=407
  train: R2=0.4795  MAPE=33.5%
  test:  R2=0.1689  RMSE=1,474.52  MAE=601.81  MAPE=44.3%  (trivial baseline MAPE=129.9%)
  gap:   R2 diff=0.3106   MAPE diff=10.8pp


## Verdict

Running `python3 ml/training/train.py` reproduces these numbers exactly (same split mechanism, same `random_state=42`, confirmed identically to the pattern established in `notebooks/03_baseline_model.ipynb`).

Honest comparison against the task 3 baseline (which trained on the leakage-prone feature):

| | Venta (task 3 → task 6) | Alquiler (task 3 → task 6) |
|---|---|---|
| test R² | 0.76 → 0.46 | 0.53 → 0.17 |
| test MAPE | 15.6% → 41.3% | 18.0% → 44.3% |
| train/test R² gap | 0.20 → 0.12 | 0.45 → 0.31 |

The definitive model is meaningfully less accurate than the baseline, but meaningfully less overfit — that's the real cost of removing leakage that was previously inflating both numbers, not a regression to hide. Both models still clearly beat the trivial baseline (predict-the-mean MAPE of 117%/130%), just by a smaller margin than before. Alquiler's single-split test R² (0.17) sits toward the low end of the CV fold spread measured above (roughly 0.17–0.55) — expected single-split noise on a model this size, not a new problem, and the same `random_state=42` convention is kept deliberately so this number is directly comparable to task 3's, not cherry-picked.